In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\mahar\Capstone-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
212,"Awwww....yes, it is heartwarming and all that ...",negative
99,"I am a Shakespeare fan, and I can appreciate w...",negative
331,"I don't go for that many ""heist"" comedies, and...",positive
186,I hired the DVD yesterday and first of all it ...,negative
185,It is a tricky thing to play a queen. On the o...,positive


In [5]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\mahar\AppData\Local\Temp\ipykernel_8796\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [6]:
df = normalize_text(df)
df.head()

,review,sentiment
212,awwww yes heartwarming unlucky family get adop...,negative
99,shakespeare fan appreciate ken branagh done br...,negative
331,go many heist comedy might care one actor made...,positive
186,hired dvd yesterday first started bad aspect r...,negative
185,tricky thing play queen one hand actress majes...,positive


In [7]:
df['sentiment'].value_counts()

sentiment
negative    261
positive    239
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
212,awwww yes heartwarming unlucky family get adop...,0
99,shakespeare fan appreciate ken branagh done br...,0
331,go many heist comedy might care one actor made...,1
186,hired dvd yesterday first started bad aspect r...,0
185,tricky thing play queen one hand actress majes...,1


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/chinmayeeM220/Capstone-Project.mlflow')
dagshub.init(repo_owner='chinmayeeM220', repo_name='Capstone-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as chinmayeeM220

Initialized MLflow to track repo "chinmayeeM220/Capstone-Project"

Repository chinmayeeM220/Capstone-Project initialized!

<Experiment: artifact_location='mlflow-artifacts:/d0a24f7d31f1496cba2aff99b4a345cf', creation_time=1774195965592, experiment_id='0', last_update_time=1774195965592, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [14]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-03-31 10:49:26,320 - INFO - Starting MLflow run...
2026-03-31 10:49:27,528 - INFO - Logging preprocessing parameters...
2026-03-31 10:49:28,645 - INFO - Initializing Logistic Regression model...
2026-03-31 10:49:28,648 - INFO - Fitting the model...
2026-03-31 10:49:28,727 - INFO - Model training complete.
2026-03-31 10:49:28,731 - INFO - Logging model parameters...
2026-03-31 10:49:29,248 - INFO - Making predictions...
2026-03-31 10:49:29,250 - INFO - Calculating evaluation metrics...
2026-03-31 10:49:29,266 - INFO - Logging evaluation metrics...
2026-03-31 10:49:31,017 - INFO - Saving and logging the model...
2026/03/31 10:49:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 10:49:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run stately-vole-987 at: https://dagshub.com/chinmayeeM220/Capstone-Project.mlflow/#/experiments/0/runs/b2272ba4289d4be0a15e52c6fe4bf8f8
🧪 View experiment at: https://dagshub.com/chinmayeeM220/Capstone-Project.mlflow/#/experiments/0
